In [ ]:
import sys
import argparse
import logging
import json
import pandas as pd
import numpy as np
import h5py
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional
import re
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

from src.util.log_manager import setup_logging
from src.simulation.run_emates import run_parallel_emates_simulations
from src.util.path_manager import get_paths
from data_loader import DataLoader

In [ ]:
class EMATESPipeline:
    """eMATESシミュレーションの統合パイプライン - データ整理特化版"""
    
    def __init__(self, parallel_workers: int = 4, debug: bool = False):
        self.parallel_workers = parallel_workers
        self.debug = debug
        self.logger = self._setup_logging()
        self.loader = DataLoader()
        self.scenario_data = {}
        self.optimization_results = {}  # 最適化結果保存用
        
    
    
    def _evaluate_performance(self, yearly_data: Dict) -> Dict:
        """性能評価"""
        metrics = {
            "profit": 0,
            "waiting_time": 0,
            "loss_count": 0,
            "utilization": 0
        }
        
        for worker_name, data in yearly_data["cs_data"].items():
            initial_cost = self._initial_cost(data)
            
            # 待機時間計算
            vehicle_trip = data['vehicle_trip']
            non_zero = vehicle_trip[vehicle_trip['WaitingEntryTime'] != 0]
            if not non_zero.empty:
                waiting_times = non_zero['startChargingTime'].values - non_zero['WaitingEntryTime'].values
                avg_waiting = waiting_times[waiting_times > 0].mean()
            else:
                avg_waiting = 0
            
            # ロス回数
            loss_count = len(data['charging_loss'])
            
            metrics['total_cost'] += initial_cost
            metrics["waiting_time"] += avg_waiting
            metrics["loss_count"] += loss_count

        
        return metrics
    
    def _initial_cost(self, data: Dict) -> float:
        """初期コスト計算（単純な例）"""
        # 出力に応じて初期コストを計算
        if 'ports' not in data or not data['ports']:
            return 0.0
        total_cost = sum(data['ports']) * 1000
        return total_cost
    
    def _optimize_cs_configuration(self, yearly_data: Dict, performance: Dict) -> Dict:
        """MILP最適化（簡略版）"""
        # 現在の設定を基準に改善案を生成
        current_config = yearly_data["cs_data"]["worker_1"]
        
        # 簡単な最適化ロジック（実際にはMILPソルバーを使用）
        if performance["waiting_time"] > 100:  # 待機時間が長い場合
            # ポート数を増加
            new_ports = [p + 1 for p in current_config["ports"]]
        else:
            # 現状維持
            new_ports = current_config["ports"]
        
        optimal_config = {
            "csids": current_config["csids"],
            "ports": new_ports,
            "cap_kw": current_config["cap_kw"]
        }
        
        return optimal_config
    
    def _update_cs_configuration(self, scenario_id: int, year: int, config: Dict):
        """CS設定更新（次年度用）"""
        # 実際にはeMATESの設定ファイルを更新
        self.logger.info(f"シナリオ{scenario_id} - {year}年目のCS設定更新")
        # TODO: csList.txtファイルの更新処理
    
    def _initialize_cs_configuration(self, scenario_id: int):
        """初期CS設定"""
        self.logger.info(f"シナリオ{scenario_id}の初期CS設定")
        # TODO: 初期設定の定義
    
    # ===== 結果分析・可視化 =====
    def get_optimization_summary(self) -> pd.DataFrame:
        """最適化結果のサマリー取得"""
        summary_data = []
        
        for key, result in self.optimization_results.items():
            summary_data.append({
                "scenario_id": result["scenario_id"],
                "year": result["year"],
                "profit": result["performance"]["profit"],
                "waiting_time": result["performance"]["waiting_time"],
                "loss_count": result["performance"]["loss_count"],
                "total_ports": sum(result["optimal_config"]["ports"])
            })
        
        return pd.DataFrame(summary_data)
    
    def save_results(self, filepath: str = None):
        """結果保存"""
        if filepath is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filepath = f"optimization_results_{timestamp}.json"
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.optimization_results, f, ensure_ascii=False, indent=2, default=str)
        
        self.logger.info(f"最適化結果を保存: {filepath}")
        
    def _setup_logging(self):
        """ログ設定"""
        setup_logging(self.debug)
        return logging.getLogger(__name__)
    
    def load_cs_configurations(self, config_path: str) -> List[Dict]:
        """CS配置設定を読み込み"""
        try:
            with open(config_path, 'r') as f:
                solutions = json.load(f)
            self.logger.info(f"CS配置設定を読み込み: {len(solutions)}個の設定")
            return solutions
        except (ValueError, FileNotFoundError) as e:
            self.logger.error(f"設定読み込みエラー: {e}")
            raise
    
    def run_simulations(self, solutions: List[Dict]) -> int:
        """シミュレーション実行"""
        self.logger.info(f"シミュレーション開始: {len(solutions)}個の設定")
        
        # 各設定を順次実行（並列化も可能）
        for i, solution in enumerate(solutions):
            self.logger.info(f"設定 {i+1}/{len(solutions)} を実行中...")
            result = run_parallel_emates_simulations(solution, self.parallel_workers)
            if result != 0:
                self.logger.error(f"設定 {i+1} の実行に失敗")
                return result
        
        self.logger.info("全シミュレーション完了")
        return 0
    
    def run_data_collection_pipeline(self, config_path: Optional[str] = None) -> Dict[str, Dict]:
        """データ収集パイプライン実行"""
        try:
            # シミュレーション実行（設定ファイルが指定された場合のみ）
            if config_path:
                solutions = self.load_cs_configurations(config_path)
                sim_result = self.run_simulations(solutions)
                if sim_result != 0:
                    raise RuntimeError("シミュレーション実行に失敗")
            
            # データ収集と整理
            worker_data = self.collect_worker_data()
            
            if worker_data:
                self.logger.info("データ収集パイプライン完了")
                return worker_data
            else:
                self.logger.warning("収集されたデータが空です")
                return {}
            
        except Exception as e:
            self.logger.error(f"データ収集パイプライン実行エラー: {e}")
            return {}

In [54]:
# ===== Step 1: データ収集・基本情報取得 =====
pipeline = EMATESPipeline()
scenario_id = 1
year = 0
worker_ids = [1]

# 現在のCS設定を取得
paths = get_paths(1)
result_dir = Path(paths["result"])
worker_data = pipeline._load_worker_data(1, result_dir)

# 基本情報
csids = worker_data['csids']
current_ports = worker_data['ports'] 
cap_kw = worker_data['cap_kw']
total_ports = worker_data['total_ports']

print(f"現在のCS設定: {len(csids)}箇所, 総ポート数: {total_ports}")
print(f"CSID: {csids}")
print(f"ポート数: {current_ports}")

2025-06-10 12:44:30,306 - INFO - ログ記録を開始しました: C:\Users\echiz\00_研究コード\eMATES解析_GA\emates\src\util\logs\simulation_20250610_124430.log
2025-06-10 12:44:34,710 - INFO - datetime変換完了: 1440行, 期間: 2024-01-01 00:01:00 - 2024-01-02 00:00:00
2025-06-10 12:44:34,718 - INFO - datetime変換完了: 1440行, 期間: 2024-01-01 00:01:00 - 2024-01-02 00:00:00
2025-06-10 12:44:34,728 - INFO - datetime変換完了: 6行, 期間: 2024-01-01 08:37:00 - 2024-01-01 10:09:00
2025-06-10 12:44:34,778 - INFO - Worker 1: CS情報取得完了 - 9箇所, 総ポート数: 11


現在のCS設定: 9箇所, 総ポート数: 11
CSID: [900000, 900002, 900004, 900006, 900008, 900010, 900012, 900014, 900016]
ポート数: [2, 2, 2, 0, 1, 1, 2, 0, 1]


In [ ]:



# ===== Step 3: コスト計算（ベタ書き） =====
charger_cost = {50: 380, 90: 670, 100: 730}  # 万円/ポート
substation_per_kW = 2  # 万円/kW

total_charger_cost = 0
total_capacity = 0

for i, (capacity, ports) in enumerate(zip(cap_kw, current_ports)):
    if capacity <= 50:
        unit_cost = charger_cost[50]
    elif capacity <= 90:
        unit_cost = charger_cost[90]
    else:
        unit_cost = charger_cost[100]
    
    charger_cost_per_cs = unit_cost * ports
    total_charger_cost += charger_cost_per_cs
    total_capacity += capacity * ports
    
    print(f"CS{csids[i]}: {capacity}kW×{ports}ポート = {charger_cost_per_cs}万円")

substation_cost = total_capacity * substation_per_kW
total_initial_cost = total_charger_cost + substation_cost

print(f"充電器コスト: {total_charger_cost}万円")
print(f"変電所コスト: {substation_cost}万円") 
print(f"総初期費用: {total_initial_cost}万円")

# ===== Step 4: 評価値計算（ベタ書き） =====
# 重み設定
weight_cost = 0.4
weight_waiting = 0.3
weight_loss = 0.2
weight_util = 0.1

# 正規化（0-100スケール）
normalized_cost = (total_initial_cost / 1000)  # 1000万円を基準
normalized_waiting = min(avg_waiting / 100, 100)  # 100秒を上限
normalized_loss = min(loss_rate, 100)
normalized_util_penalty = max(0, 50 - avg_utilization)  # 50%以下はペナルティ

# 総合評価値（小さいほど良い）
total_score = (
    normalized_cost * weight_cost +
    normalized_waiting * weight_waiting + 
    normalized_loss * weight_loss +
    normalized_util_penalty * weight_util
)

print(f"総合評価値: {total_score:.2f}")

# ===== Step 5: 改善判定（ベタ書き） =====
print("\n=== 改善提案 ===")

# 条件別の改善提案
if avg_waiting > 300:  # 5分以上
    print("⚠️ 待機時間が長い → ポート数増加を検討")
    suggested_action = "increase_ports"
elif loss_rate > 10:  # 10%以上
    print("⚠️ ロス率が高い → CS数増加を検討") 
    suggested_action = "add_cs"
elif avg_utilization < 30:  # 30%未満
    print("⚠️ 利用率が低い → ポート数削減を検討")
    suggested_action = "decrease_ports"
else:
    print("✅ 現状維持")
    suggested_action = "maintain"

# ===== Step 6: 設定更新（ベタ書き） =====
new_ports = current_ports.copy()

if suggested_action == "increase_ports":
    # 待機時間が最も長いCSのポートを増加
    # (実際は時系列データから特定)
    worst_cs_idx = 0  # 仮
    new_ports[worst_cs_idx] = min(new_ports[worst_cs_idx] + 1, 4)
    print(f"CS{csids[worst_cs_idx]}のポート数: {current_ports[worst_cs_idx]} → {new_ports[worst_cs_idx]}")

elif suggested_action == "decrease_ports":
    # 利用率が最も低いCSのポートを削減
    best_cs_idx = -1  # 仮
    new_ports[best_cs_idx] = max(new_ports[best_cs_idx] - 1, 1)
    print(f"CS{csids[best_cs_idx]}のポート数: {current_ports[best_cs_idx]} → {new_ports[best_cs_idx]}")

print(f"更新後設定: {dict(zip(csids, new_ports))}")

# ===== Step 7: ファイル更新（ベタ書き） =====
# csList.txtの更新
cs_data_new = []
for csid, port, capacity in zip(csids, new_ports, cap_kw):
    cs_data_new.append([csid, port, capacity])

# 新しいcsListファイル作成
df_new = pd.DataFrame(cs_data_new, columns=['CSID', 'Port', 'Cap_kw'])
csList_path = Path(paths["csList"]) 
df_new.to_csv(csList_path, index=False, header=False)

print(f"csList.txt更新完了: {csList_path}")
print("次年度シミュレーション準備完了")

In [4]:
pipeline = EMATESPipeline(parallel_workers=4, debug=True)
result_year_dict = pipeline.run_yearly_optimization(scenario_id=1, year=0, worker_ids=[1])

2025-06-10 11:19:52,757 - INFO - ログ記録を開始しました: C:\Users\echiz\00_研究コード\eMATES解析_GA\emates\src\util\logs\simulation_20250610_111952.log
2025-06-10 11:19:52,763 - INFO - シナリオ1 - 0年目の最適化開始


2025-06-10 11:19:57,625 - INFO - CS情報の統一: Cap_kW (1440, 23)
2025-06-10 11:19:57,636 - INFO - datetime変換完了: 1440行, 期間: 2024-01-01 00:01:00 - 2024-01-02 00:00:00
2025-06-10 11:19:57,651 - INFO - datetime変換完了: 6行, 期間: 2024-01-01 08:37:00 - 2024-01-01 10:09:00
2025-06-10 11:19:57,705 - INFO - Worker 1: CS情報取得完了 - 9箇所, 総ポート数: 11


AttributeError: 'EMATESPipeline' object has no attribute '_calculate_revenue'

In [ ]:
# パイプライン実行後、そのまま分析
pipeline = EMATESPipeline()
df = pipeline.collect_scenario_data(scenario_id=1, worker_ids=[1])



2025-06-09 19:04:49,931 - INFO - ログ記録を開始しました: C:\Users\echiz\00_研究コード\eMATES解析_GA\emates\src\util\logs\simulation_20250609_190449.log
2025-06-09 19:04:54,194 - INFO - CS情報の統一: Cap_kW (1440, 23)
2025-06-09 19:04:54,194 - INFO - datetime変換完了: 1440行, 期間: 2024-01-01 00:01:00 - 2024-01-02 00:00:00
2025-06-09 19:04:54,213 - INFO - datetime変換完了: 6行, 期間: 2024-01-01 08:37:00 - 2024-01-01 10:09:00
2025-06-09 19:04:54,260 - INFO - Worker 1: CS情報取得完了 - 9箇所, 総ポート数: 11


In [11]:
from src.util.scenario import get_adoption_rate

ImportError: cannot import name 'get_adoption_rate' from 'src.util.scenario' (c:\Users\echiz\00_研究コード\eMATES解析_GA\emates\src\util\scenario.py)

In [ ]:
# 辞書の中身を確認する基本的な方法
print(list(df.keys()))  # リスト形式で表示
print(df.get('scenario_id'))  # キーが存在しない場合はNoneを返す
# 3. 階層構造の可視化
import json
print(json.dumps(df, indent=2, default=str))  # 日付などもstr変換


['plan_id', 'metadata', 'cs_data']
1
{
  "plan_id": 1,
  "metadata": {
    "created_at": "2025-06-09 19:04:49.932945",
    "worker_count": 1
  },
  "cs_data": {
    "worker_1": {
      "worker_id": 1,
      "result_dir": "\\\\wsl.localhost\\ubuntu-22.04\\home\\tsato-cnlab\\Emates\\eMATES_2308\\network\\coupled_network_shikata_1\\result",
      "timeseries": "                     ElapsedTime  90000000  90000001  90000200  90000201  \\\nTimestamp                                                                  \n2024-01-01 00:01:00           60       0.0       0.0       0.0       0.0   \n2024-01-01 00:02:00          120       0.0       0.0       0.0       0.0   \n2024-01-01 00:03:00          180      90.0       0.0       0.0       0.0   \n2024-01-01 00:04:00          240      90.0       0.0       0.0       0.0   \n2024-01-01 00:05:00          300      90.0       0.0       0.0       0.0   \n...                          ...       ...       ...       ...       ...   \n2024-01-01 23:56:00   

In [29]:
# 直接評価・分析
for worker_name, data in worker_data.items():
    timeseries = data['timeseries']
    charging_loss = data['charging_loss']
    vehicle_trip = data['vehicle_trip']
    
    # 評価指標計算
    total_capacity = timeseries['cap_kw'].sum().sum()
    non_zero = vehicle_trip[vehicle_trip['WaitingEntryTime'] != 0]
    waiting_times = non_zero['startChargingTime'].values - non_zero['WaitingEntryTime'].values

    
    # 待機時間の統計
    avg_waiting = waiting_times.mean()
    max_waiting = waiting_times.max()
    min_waiting = waiting_times.min()
    total_waiting_events = len(waiting_times[waiting_times > 0])  # 待機が発生した回数

    
    loss_count = len(charging_loss) if not charging_loss.empty else 0
    
    print(f"{worker_name}:")
    print(f"  容量: {total_capacity:.2f}kWh")
    print(f"  平均待機時間: {avg_waiting:.2f}ms")
    print(f"  最大待機時間: {max_waiting:.2f}ms") 
    print(f"  待機発生回数: {total_waiting_events}回")
    print(f"  充電ロス: {loss_count}回")
    print("-" * 40)

worker_1:
  容量: 62453970.00kWh
  平均待機時間: 22862.96ms
  最大待機時間: 425700.00ms
  待機発生回数: 2回
  充電ロス: 6回
----------------------------------------
worker_2:
  容量: 62444700.00kWh
  平均待機時間: 227735.56ms
  最大待機時間: 3931700.00ms
  待機発生回数: 12回
  充電ロス: 25回
----------------------------------------


In [51]:
18.17e-04

0.001817

In [ ]:
def objective()-> float:
    """目的関数の定義
    事業者の利潤最大化を目的とする"""
    init_cost = _initial_cost()
    op_cost = _operating_cost()
    
    benefit = 
    return benefit

def _initial_cost(cs_configuration: List[Dict])-> float:
    """充電器の初期費用計算
    
    Args:
        cs_configuration: CS配置設定のリスト
                         [{'id': CS_ID, 'capacity': 充電容量(kW), 'location': (x, y)}, ...]
    
    Returns:
        float: 初期費用総額（万円）
    """
    charger_cost = {'50kW': 380, '90kW':670,'100kW': 730} # kWあたりのコスト：万円
    installation_cost = 0 # 設置コストは未定義
    substation_per_kW = 2 # 変電所のkWあたりのコスト：万円/kW
    
    total_charger_cost = 0      # 充電器コスト
    total_installation_cost = 0 # 設置コスト
    total_substation_cost = 0          # 総容量
    
    for cs in cs_configuration:
        capacity = cs.get('capacity', 50)  # デフォルト50kW
        num_ports = cs.get('num_ports', 0)  # ポート数（デフォルト0）
        
        
        # 充電器種別を判定
        if capacity <= 50:
            charger_type = '50kW'
        elif capacity <= 90:
            charger_type = '90kW'
        else:
            charger_type = '100kW'
        
        # 充電器コスト計算
        unit_cost = charger_cost[charger_type]
        total_charger_cost += unit_cost
        
        # 設置コスト計算
        total_installation_cost += installation_cost
        
        # 総容量累計
        total_capacity += capacity
    
    # 変電所コスト計算
    substation_cost = total_capacity * substation_per_kW
    
    # 初期費用合計
    initial_cost = total_charger_cost + total_installation_cost + substation_cost
    
    return initial_cost

def _operating_cost(df_cap_kw: pd.DatFrame, operation_months: int = 12)-> float:
    """運用コスト計算
    
    Args:
        df_cap_kw: 一日の時系列データ（DataFrame）
        operation_months: 運用期間（月数）
    
    Returns:
        float: 運用コスト（万円）
    """
    electricity_basic_fee = 1911.8e-04  # 基本料金：万円/kW・月
    electricity_use_fee = 18.07e-04     # 従量料金：万円/kWh
    
    # 一日の総充電量（kWh）を計算
    total_energy_kwh = df_cap_kw.sum().sum() /60  # 1日の総容量をkWhに変換
    max_capacity = df_cap_kw.max()  # 最大容量（kW）
    
    # 基本料金（容量分）
    basic_cost = electricity_basic_fee * max_capacity * operation_months
    # 従量料金（使用電力分）
    usage_cost = electricity_use_fee * total_energy_kwh * operation_months * 365  # 年間の使用電力分
    
    # 運用コスト合計
    operating_cost = basic_cost + usage_cost
    
    return operating_cost


In [52]:
cs_config = {
    "csid": [1,2,3],
    "charger_type": ["50kW","90kW","100kW"],
    "num_ports": [2, 1, 1]
}

In [19]:
non_zero = vehicle_trip[vehicle_trip['WaitingEntryTime'] != 0]
non_zero.head(1000)

,EVID,StartTime,EndTime,WaitingEntryTime,startChargingTime,startID,goalID,tripLength,CSID,InitialSOC
159,7,21800,1742100,159100,159100,5001,1437,5447.07,900000.0,0.2
481,1181,3199100,4963100,3379100,3379100,5001,1437,5447.09,900000.0,0.2
967,2897,12808900,14519300,12955100,12955100,5001,169,5472.08,900000.0,0.2
1545,4480,22723100,24432900,22868700,22868700,5001,169,5472.09,900000.0,0.2
2687,6904,26488200,28882000,26765700,26765700,5003,1123,9112.89,900002.0,0.2
...,...,...,...,...,...,...,...,...,...,...
19286,63377,81610400,83911800,81811100,81811100,5005,1031,9099.73,900006.0,0.2
19314,63881,82313600,84091500,82525800,82525800,5001,169,5472.96,900000.0,0.2
19419,64252,82825100,84745600,83060400,83060400,5003,169,6448.70,900002.0,0.2
19642,64961,84305200,******,84456500,84456500,5001,1123,5709.22,0.0,NaN
